# HoloGuide: AI sees, rules decide, people oversee

This notebook (DOC-04a skeleton) runs one HoloGuide scenario from start to finish, offline:

1. A PyBullet scene renders the *Sealed Cover Assembly* (base plate → gasket → cover → bolts).
2. An **inspector** looks at the rendered image only and reports an `Observation` (evidence, not a decision).
3. Deterministic **rules** turn the Observation into a `Decision` (pass / block / needs review / hold).
4. The **workflow** advances only on a pass (or a logged operator override), and every event goes to an audit log.

The default inspector is OpenCV (no GPU, no model). Set `INSPECTOR = "hybrid"` or `"gemma"` if a local Ollama
with `gemma3:4b` is already running; the notebook never downloads a model.

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config" / "assembly.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import load_config
config = load_config()
print(config.name, "-", len(config.steps), "steps; confidence threshold", config.confidence_threshold)
for step in config.steps:
    print(f"  step {step.id}: {step.name} (adds {', '.join(step.adds_parts)})")

## 1. Choose an inspector

All inspectors share one contract: `inspect(image_path, step) -> Observation`. They never raise and never see
the simulator's ground truth.

In [ ]:
from src.vision import factories

INSPECTOR = os.environ.get("HOLOGUIDE_NOTEBOOK_INSPECTOR", "cv")   # cv | hybrid | gemma
inspector = {"cv": factories.cv, "hybrid": factories.hybrid, "gemma": factories.gemma}[INSPECTOR](config)
print("Using", INSPECTOR, "->", type(inspector).__name__)

## 2. Inspect one image by hand

Render the *missing gasket* mistake (cover placed straight on the base plate) and check it against step 3.

In [ ]:
from datetime import datetime
from src.simulation.assembly_env import AssemblyEnv
from src.control.rules import RuleContext, evaluate

RUN_DIR = ROOT / "outputs" / "notebook" / datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=False)

with AssemblyEnv(config, seed=0) as env:
    for part in ("base_plate", "cover"):          # the gasket is missing
        env.place_part(part)
    frame = env.render_to_file(str(RUN_DIR / "missing_gasket.png"))

step = config.step(3)
observation = inspector.inspect(frame, step)       # only the image path and the step
decision = evaluate(observation, step, config.confidence_threshold,
                    context=RuleContext.from_config(config, step))
print("Observation:", observation.model_dump_json(indent=2))
print("Decision:", decision.status.value, "-", decision.message, f"(decided by {decision.decided_by.value})")

In [ ]:
try:
    from IPython.display import display
    from PIL import Image
    display(Image.open(frame))
except ImportError:
    print("Image saved at", frame)

## 3. Run a full scenario through the workflow

`ScenarioRunner` plays the scripted operator actions: the mistake is injected, the step is **blocked**, the operator
fixes it, the step is re-inspected and the workflow continues to **Complete**.

In [ ]:
from src.control.audit_log import AuditLog
from src.control.workflow import Workflow
from src.simulation.runner import ScenarioRunner

SCENARIO = "missing_gasket"     # correct | missing_gasket | wrong_cover | wrong_order
audit = AuditLog(RUN_DIR / "audit.jsonl")
with AssemblyEnv(config, seed=0) as env:
    workflow = Workflow(config, inspector, audit)
    result = ScenarioRunner(SCENARIO, env, workflow, RUN_DIR / "frames").run()
print(result)
assert result.completed, result.error

In [ ]:
frames = sorted((RUN_DIR / "frames").glob("*.png"))
try:
    from IPython.display import display
    from PIL import Image
    thumbs = [Image.open(f).resize((320, 240)) for f in frames]
    strip = Image.new("RGB", (320 * len(thumbs), 240), "white")
    for i, thumb in enumerate(thumbs):
        strip.paste(thumb, (320 * i, 0))
    display(strip)
except ImportError:
    pass
for f, status in zip(frames, result.decisions):
    print(f.name, "->", status)

## 4. The audit log

Every inspection, decision, correction and override is one JSON line.

In [ ]:
import pandas as pd

events = audit.read_all()
table = pd.DataFrame([{
    "event": e["event_type"],
    "step": (e.get("current_step") or {}).get("id"),
    "state": e.get("next_state"),
    "decision": (e.get("decision") or {}).get("status"),
    "confidence": (e.get("observation") or {}).get("confidence"),
    "detail": (e.get("decision") or {}).get("message") or e.get("reason") or e.get("part") or "",
} for e in events])
table

## 5. Evaluation results

Measured numbers come from `src.evaluation.runner` runs saved under `outputs/evaluation/` (see `docs/decisions.md`).
Anything not measured on this machine is shown as N/A, never estimated.

In [ ]:
rows = []
for label, stem in [("OpenCV", "cv-v2-run1"), ("Gemma 3 4B", "gemma-v2-run1"), ("Hybrid", "hybrid-v2-run1")]:
    path = ROOT / "outputs" / "evaluation" / f"{stem}.summary.json"
    if not path.exists():
        rows.append({"inspector": label, "note": "N/A (run not found)"})
        continue
    s = json.loads(path.read_text())
    row = {"inspector": label, "false_block_rate": s.get("false_block_rate"),
           "median_latency_s": (s.get("latency") or {}).get("median_s")}
    for kind, metric in (s.get("accuracy_by_mistake_type") or {}).items():
        row[f"acc_{kind}"] = metric.get("accuracy")
    rows.append(row)
pd.DataFrame(rows)